# Training notebook for MLP

This notebook implements training for MLP architectures for image classification.



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [2]:
import os
import random
import csv
import time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from tqdm import tqdm
import time

from PIL import Image, ImageFilter
import cv2


## Reproducibility

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cuda


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
zip_path = "/content/drive/MyDrive/DLprojekt1/archive.zip"
!unzip -q "/content/drive/MyDrive/DLprojekt1/archive.zip" -d "/content/"

## Paths

In [6]:
# global variables and creating folders
DATA_DIR = "/content"

PROJECT_DIR = "/content/drive/MyDrive/DLprojekt1"

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
PLOT_DIR = os.path.join(PROJECT_DIR, "plots")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

RESULT_FILE = os.path.join(RESULT_DIR, "experiment_results.csv")

if not os.path.exists(RESULT_FILE):

    with open(RESULT_FILE,"w",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            "experiment",
            "lr",
            "optimizer",
            "dropout",
            "weight_decay",
            "augmentation",
            "test_accuracy"
        ])


## Data Augmentations

In [7]:

class SobelAugmentation:

    def __init__(self, alpha=0.5):
        self.alpha = alpha

    def __call__(self, img):

        img_np = np.array(img)

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        sobel = np.sqrt(sobelx**2 + sobely**2)

        sobel = np.uint8(255 * sobel / np.max(sobel))

        sobel_rgb = np.stack([sobel]*3, axis=-1)

        blended = cv2.addWeighted(img_np, 1-self.alpha, sobel_rgb, self.alpha, 0)

        return Image.fromarray(blended)


class GaussianBlur(object):

    def __call__(self,img):

        return img.filter(ImageFilter.GaussianBlur(radius=1.0))


def get_transform(augmentation=None):

    transform_list = []

    if augmentation == "rotation":
        transform_list.append(transforms.RandomRotation(15))

    if augmentation == "color_jitter":
        transform_list.append(
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            )
        )

    if augmentation == "gaussian_blur":
        transform_list.append(GaussianBlur())

    if augmentation == "sobel":
        transform_list.append(SobelAugmentation())

    transform_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4789, 0.4723, 0.4305),
            (0.2421, 0.2383, 0.2587)
        )
    ])

    return transforms.Compose(transform_list)


def mixup_data(x, y, alpha=0.2):

    lam = np.random.beta(alpha,alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam*x + (1-lam)*x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

def mixup_loss(criterion, pred, y_a, y_b, lam):

    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)


## Data Loading

In [8]:
def load_data(augmentation=None, batch_size=256, subset_ratio=1.0):

    train_transform = get_transform(augmentation)

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4789, 0.4723, 0.4305),
            (0.2421, 0.2383, 0.2587)
        )
    ])

    train_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "train"),
        transform=train_transform
    )

    valid_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "valid"),
        transform=test_transform
    )

    test_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "test"),
        transform=test_transform
    )

    if subset_ratio < 1.0:

        subset_size = int(len(train_set)*subset_ratio)
        train_set = torch.utils.data.Subset(train_set, range(subset_size))

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, valid_loader, test_loader


## MLP Architecture

In [9]:
class MLP(nn.Module):

    def __init__(self, dropout):

        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32*32*3, 384)
        self.bn1 = nn.BatchNorm1d(384)

        self.fc2 = nn.Linear(384, 192)
        self.bn2 = nn.BatchNorm1d(192)

        self.fc3 = nn.Linear(192, 10)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        x = self.flatten(x)

        x = F.gelu(self.bn1(self.fc1(x)))
        x = self.dropout(x)

        x = F.gelu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        x = self.fc3(x)

        return x

## Evaluation

In [10]:

def evaluate(model, loader):

    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_correct / total, total_loss / total

In [11]:
class EarlyStopping:

    def __init__(self,patience=5):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def step(self,score):

        if self.best_score is None:
            self.best_score = score

        elif score <= self.best_score:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

        else:

            self.best_score = score
            self.counter = 0

        return self.stop

## Training

In [12]:
def train_model(model, train_loader, val_loader, optimizer, epochs=30,
                use_mixup=False, name=None):

    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping()

    history = {
        "train_loss":[],
        "val_loss":[],
        "val_acc":[],
    }

    best_acc = 0

    for epoch in range(epochs):

        start_time = time.time()

        model.train()
        running_loss = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", unit="batch"):

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            if use_mixup:

                images,targets_a,targets_b,lam = mixup_data(images,labels)
                outputs = model(images)
                loss = mixup_loss(criterion, outputs, targets_a, targets_b, lam)

            else:

                outputs = model(images)
                loss = criterion(outputs,labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)

        val_acc, val_loss = evaluate(model, val_loader)
        train_loss = running_loss / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        epoch_time = time.time() - start_time

        print("train_loss:", f"{train_loss:.6f}", ", val_loss:", f"{val_loss:.6f}",
              ", val_acc:", f"{val_acc:.6f}", ", time:", round(epoch_time, 2), "s\n")

        torch.save(
            model.state_dict(),
            os.path.join(CHECKPOINT_DIR,f"checkpoint_epoch_{name}_{epoch+1}.pt")
        )

        if val_acc > best_acc:
            best_acc = val_acc

        if early_stopping.step(val_acc):

          print("Early stopping triggered")
          break

    return history


## Run Experiment

In [13]:
def save_history_csv(history, experiment_name):

    filename = os.path.join(RESULT_DIR, f"{experiment_name}_history.csv")

    epochs = len(history['train_loss'])

    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss', 'val_acc'])
        for i in range(epochs):
            writer.writerow([
                i+1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['val_acc'][i]
            ])

    print(f"History saved in: {filename}")

In [14]:
def run_experiment(model_class, name, lr, optimizer_name, dropout, weight_decay, augmentation=None,
                   epochs=40, subset_ratio=1.0, use_mixup=False):

    print("Running:", name ,"\n")

    train_loader, val_loader, test_loader = load_data(
        augmentation=augmentation,
        subset_ratio=subset_ratio
    )

    model = model_class(dropout).to(device)

    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    elif optimizer_name == "SGD_momentum":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    elif optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = train_model(model, train_loader, val_loader, optimizer, epochs=epochs, use_mixup=use_mixup, name=name)

    test_acc, _ = evaluate(model, test_loader)
    save_history_csv(history, name)

    torch.save(
        model.state_dict(),
        os.path.join(MODEL_DIR,f"{name}.pt")
    )

    with open(RESULT_FILE,"a",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            name,
            lr,
            optimizer_name,
            dropout,
            weight_decay,
            augmentation,
            test_acc
        ])

    print("Test accuracy:", test_acc)


## Experiments

Experiments with different training, regularization parameters and data augumentation methods

In [ ]:
run_experiment(
    model_class=MLP,
    name="mlp_lr_0.1",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

run_experiment(
    model_class=MLP,
    name="mlp_lr_0.01",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

run_experiment(
    model_class=MLP,
    name="mlp_lr_0.001",
    lr=0.001,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

Running: mlp_lr_0.1 



Epoch 1: 100%|██████████| 352/352 [00:33<00:00, 10.45batch/s]


train_loss: 1.819376 , val_loss: 1.683469 , val_acc: 0.387344 , time: 66.48 s



Epoch 2: 100%|██████████| 352/352 [00:35<00:00,  9.90batch/s]


train_loss: 1.683626 , val_loss: 1.621222 , val_acc: 0.410078 , time: 68.14 s



...

train_loss: 1.758423 , val_loss: 1.723309 , val_acc: 0.382544 , time: 65.91 s



Epoch 19: 100%|██████████| 352/352 [00:33<00:00, 10.66batch/s]


train_loss: 1.751805 , val_loss: 1.717891 , val_acc: 0.385644 , time: 64.81 s



Epoch 20: 100%|██████████| 352/352 [00:33<00:00, 10.61batch/s]


train_loss: 1.743150 , val_loss: 1.710400 , val_acc: 0.387456 , time: 66.58 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_lr_0.001_history.csv
Test accuracy: 0.3879888888888889


Best lr parameter: 0.01

In [ ]:
## to juz mamy -> lr_0.01
#run_experiment(
#    model_class=MLP,
#    name="optimizer_sgd",
#    lr=0.01,
#    optimizer_name="SGD",
#    dropout=0.3,
#    weight_decay=1e-4,
#    epochs=20
#)

run_experiment(
    model_class=MLP,
    name="mlp_optimizer_sgd_momentum",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

run_experiment(
    model_class=MLP,
    name="mlp_optimizer_adam",
    lr=0.01,
    optimizer_name="Adam",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

Running: mlp_optimizer_sgd_momentum 



Epoch 1: 100%|██████████| 352/352 [00:41<00:00,  8.51batch/s]


train_loss: 1.829725 , val_loss: 1.676586 , val_acc: 0.391056 , time: 79.96 s



Epoch 2: 100%|██████████| 352/352 [00:38<00:00,  9.14batch/s]


train_loss: 1.682547 , val_loss: 1.623994 , val_acc: 0.406878 , time: 77.28 s



...

train_loss: 1.749040 , val_loss: 1.710357 , val_acc: 0.374567 , time: 72.19 s



Epoch 8: 100%|██████████| 352/352 [00:36<00:00,  9.70batch/s]


train_loss: 1.746797 , val_loss: 1.708917 , val_acc: 0.375300 , time: 72.18 s



Epoch 9: 100%|██████████| 352/352 [00:37<00:00,  9.44batch/s]


train_loss: 1.749605 , val_loss: 1.707040 , val_acc: 0.377678 , time: 72.72 s

Early stopping triggered
History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_optimizer_adam_history.csv
Test accuracy: 0.3781555555555556


Best optimizer parameter: SGD_momentum

In [ ]:
run_experiment(
   model_class=MLP,
   name="mlp_dropout_01",
   lr=0.01,
   optimizer_name="SGD_momentum",
   dropout=0.1,
   weight_decay=1e-4,
   epochs=20
)

# run_experiment(
#     model_class=MLP,
#     name="mlp_dropout_03",
#     lr=0.1,
#     optimizer_name="SGD_momentum",
#     dropout=0.3,
#     weight_decay=1e-4,
#     epochs=20
# )

run_experiment(
    model_class=MLP,
    name="mlp_dropout_05",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.5,
    weight_decay=1e-4,
    epochs=20
)

Running: mlp_dropout_01 



Epoch 1: 100%|██████████| 352/352 [00:36<00:00,  9.71batch/s]


train_loss: 1.763935 , val_loss: 1.643773 , val_acc: 0.400722 , time: 72.75 s



Epoch 2: 100%|██████████| 352/352 [00:35<00:00,  9.80batch/s]


train_loss: 1.599549 , val_loss: 1.582435 , val_acc: 0.429533 , time: 71.1 s



...

train_loss: 1.492396 , val_loss: 1.500031 , val_acc: 0.454456 , time: 72.92 s



Epoch 19: 100%|██████████| 352/352 [00:35<00:00,  9.81batch/s]


train_loss: 1.480234 , val_loss: 1.497290 , val_acc: 0.456489 , time: 71.35 s



Epoch 20: 100%|██████████| 352/352 [00:35<00:00,  9.80batch/s]


train_loss: 1.472362 , val_loss: 1.493248 , val_acc: 0.458089 , time: 70.49 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_dropout_05_history.csv
Test accuracy: 0.4572111111111111


Best dropout parameter: 0.3


In [ ]:
run_experiment(
   model_class=MLP,
   name="mlp_weight_decay_1e2",
   lr=0.01,
   optimizer_name="SGD_momentum",
   dropout=0.3,
   weight_decay=1e-2,
   epochs=20
)

# run_experiment(
#     model_class=MLP,
#     name="mlp_weight_decay_1e3",
#     lr=0.1,
#     optimizer_name="SGD_momentum",
#     dropout=0.3,
#     weight_decay=1e-4,
#     epochs=20
# )

run_experiment(
    model_class=MLP,
    name="mlp_dweight_decay_1e5",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-5,
    epochs=20
)

Running: mlp_weight_decay_1e2 



Epoch 1: 100%|██████████| 352/352 [00:37<00:00,  9.50batch/s]


train_loss: 1.826828 , val_loss: 1.711713 , val_acc: 0.379733 , time: 72.25 s



Epoch 2: 100%|██████████| 352/352 [00:35<00:00, 10.02batch/s]


train_loss: 1.706064 , val_loss: 1.688283 , val_acc: 0.389911 , time: 69.58 s



...

train_loss: 1.315567 , val_loss: 1.471800 , val_acc: 0.471511 , time: 69.33 s



Epoch 19: 100%|██████████| 352/352 [00:36<00:00,  9.70batch/s]


train_loss: 1.303544 , val_loss: 1.473842 , val_acc: 0.470200 , time: 70.59 s



Epoch 20: 100%|██████████| 352/352 [00:35<00:00, 10.05batch/s]


train_loss: 1.291110 , val_loss: 1.480603 , val_acc: 0.470200 , time: 69.48 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_dweight_decay_1e5_history.csv
Test accuracy: 0.4703333333333333


Best weight_decay parameter: 1e-5

DATA AUGUMENTATION EXPERIMENTS

In [ ]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_color_jitter",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-5,
    augmentation="color_jitter",
    epochs=20
)

Running: mlp_augmentation_color_jitter 



Epoch 1: 100%|██████████| 352/352 [01:13<00:00,  4.76batch/s]


train_loss: 1.868153 , val_loss: 1.689217 , val_acc: 0.382644 , time: 107.82 s



Epoch 2: 100%|██████████| 352/352 [01:14<00:00,  4.72batch/s]


train_loss: 1.717780 , val_loss: 1.628163 , val_acc: 0.407644 , time: 110.29 s



...

train_loss: 1.348420 , val_loss: 1.490469 , val_acc: 0.465278 , time: 108.42 s



Epoch 19: 100%|██████████| 352/352 [01:15<00:00,  4.69batch/s]


train_loss: 1.336862 , val_loss: 1.495566 , val_acc: 0.467222 , time: 109.06 s



Epoch 20: 100%|██████████| 352/352 [01:14<00:00,  4.73batch/s]


train_loss: 1.318753 , val_loss: 1.496346 , val_acc: 0.466289 , time: 108.63 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_color_jitter_history.csv
Test accuracy: 0.4647222222222222


In [ ]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_blur",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-5,
    augmentation="gaussian_blur",
    epochs=20
)

Running: mlp_augmentation_blur 



Epoch 1: 100%|██████████| 352/352 [00:39<00:00,  8.82batch/s]


train_loss: 1.830740 , val_loss: 1.690338 , val_acc: 0.388322 , time: 74.72 s



Epoch 2: 100%|██████████| 352/352 [00:39<00:00,  8.99batch/s]


train_loss: 1.695183 , val_loss: 1.620064 , val_acc: 0.411278 , time: 72.59 s



...

train_loss: 1.442216 , val_loss: 1.483250 , val_acc: 0.463144 , time: 71.04 s



Epoch 19: 100%|██████████| 352/352 [00:38<00:00,  9.26batch/s]


train_loss: 1.432921 , val_loss: 1.469620 , val_acc: 0.471078 , time: 72.97 s



Epoch 20: 100%|██████████| 352/352 [00:37<00:00,  9.27batch/s]


train_loss: 1.424041 , val_loss: 1.475442 , val_acc: 0.468944 , time: 70.66 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_blur_history.csv
Test accuracy: 0.4698888888888889


In [15]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_sobel",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-5,
    augmentation="sobel",
    epochs=20
)

Running: mlp_augmentation_sobel 



Epoch 1: 100%|██████████| 352/352 [00:48<00:00,  7.18batch/s]


train_loss: 1.886058 , val_loss: 1.965285 , val_acc: 0.311244 , time: 82.64 s



Epoch 2: 100%|██████████| 352/352 [00:50<00:00,  7.04batch/s]


train_loss: 1.719484 , val_loss: 1.923028 , val_acc: 0.326567 , time: 82.77 s



...

train_loss: 1.229671 , val_loss: 1.836957 , val_acc: 0.388322 , time: 77.49 s



Epoch 19: 100%|██████████| 352/352 [00:47<00:00,  7.37batch/s]


train_loss: 1.210985 , val_loss: 1.855842 , val_acc: 0.382044 , time: 79.71 s



Epoch 20: 100%|██████████| 352/352 [00:46<00:00,  7.59batch/s]


train_loss: 1.189996 , val_loss: 1.869561 , val_acc: 0.387567 , time: 77.52 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_sobel_history.csv
Test accuracy: 0.38483333333333336


In [16]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_mixup",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-5,
    use_mixup=True,
    epochs=20
)

Running: mlp_augmentation_mixup 



Epoch 1: 100%|██████████| 352/352 [00:32<00:00, 10.79batch/s]


train_loss: 1.920275 , val_loss: 1.699867 , val_acc: 0.386178 , time: 64.83 s



Epoch 2: 100%|██████████| 352/352 [00:32<00:00, 10.88batch/s]


train_loss: 1.812675 , val_loss: 1.641254 , val_acc: 0.409156 , time: 63.45 s



...

train_loss: 1.558752 , val_loss: 1.486417 , val_acc: 0.464467 , time: 69.42 s



Epoch 19: 100%|██████████| 352/352 [00:34<00:00, 10.10batch/s]


train_loss: 1.545099 , val_loss: 1.479091 , val_acc: 0.468556 , time: 68.33 s



Epoch 20: 100%|██████████| 352/352 [00:35<00:00,  9.84batch/s]


train_loss: 1.546976 , val_loss: 1.480712 , val_acc: 0.467767 , time: 69.61 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_mixup_history.csv
Test accuracy: 0.4650666666666667


In [17]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_rotation",
    lr=0.01,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-5,
    augmentation="rotation",
    epochs=20
)

Running: mlp_augmentation_rotation 



Epoch 1: 100%|██████████| 352/352 [00:42<00:00,  8.32batch/s]


train_loss: 1.860050 , val_loss: 1.696892 , val_acc: 0.384144 , time: 75.67 s



Epoch 2: 100%|██████████| 352/352 [00:42<00:00,  8.24batch/s]


train_loss: 1.725277 , val_loss: 1.638940 , val_acc: 0.401422 , time: 76.32 s



...

train_loss: 1.489503 , val_loss: 1.472462 , val_acc: 0.470133 , time: 73.15 s



Epoch 19: 100%|██████████| 352/352 [00:38<00:00,  9.13batch/s]


train_loss: 1.486149 , val_loss: 1.466419 , val_acc: 0.470867 , time: 72.0 s



Epoch 20: 100%|██████████| 352/352 [00:39<00:00,  9.01batch/s]


train_loss: 1.476425 , val_loss: 1.453840 , val_acc: 0.476778 , time: 70.89 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_rotation_history.csv
Test accuracy: 0.4735888888888889
